In [42]:
!pip -q install torch transformers evaluate datasets

In [3]:
from datasets import load_dataset
data = load_dataset('sh0416/ag_news')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/33.7M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [4]:
data.column_names

{'train': ['label', 'title', 'description'],
 'test': ['label', 'title', 'description']}

#Label_names

0 = World
1 = Sports
2 = Business
3 = Sci/Tech



In [5]:
data['train'].shape

(120000, 3)

In [6]:
data['test'][100]

{'label': 2,
 'title': 'Olympic history for India, UAE',
 'description': 'An Indian army major shot his way to his country #39;s first ever individual Olympic silver medal on Tuesday, while in the same event an member of Dubai #39;s ruling family became the first ever medallist from the United Arab Emirates. '}

In [7]:
data['train'][10000]

{'label': 1,
 'title': 'A Daily Look at U.S. Iraq Military Deaths (AP)',
 'description': 'AP - As of Wednesday, Aug. 25, 964 U.S. service members have died since the beginning of military operations in Iraq in March 2003, according to the Defense Department. Of those, 722 died as a result of hostile action and 242 died of non-hostile causes.'}

In [8]:
data = data.map(lambda x:{'label':x['label']-1})

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [9]:
data['train'][10000]

{'label': 0,
 'title': 'A Daily Look at U.S. Iraq Military Deaths (AP)',
 'description': 'AP - As of Wednesday, Aug. 25, 964 U.S. service members have died since the beginning of military operations in Iraq in March 2003, according to the Defense Department. Of those, 722 died as a result of hostile action and 242 died of non-hostile causes.'}

In [10]:
data['test'][100]

{'label': 1,
 'title': 'Olympic history for India, UAE',
 'description': 'An Indian army major shot his way to his country #39;s first ever individual Olympic silver medal on Tuesday, while in the same event an member of Dubai #39;s ruling family became the first ever medallist from the United Arab Emirates. '}

In [11]:
from transformers import AutoTokenizer

In [29]:
tokenizer = AutoTokenizer.from_pretrained(
    # "bert-base-uncased"
   "roberta-base"
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [30]:
demo_text = "Hi this is Omm Tripathi"
tokenizer.tokenize(demo_text)
# bert uses sub word tokenization

['Hi', 'Ġthis', 'Ġis', 'ĠO', 'mm', 'ĠTrip', 'athi']

In [31]:
tokens = tokenizer(demo_text)
tokens

{'input_ids': [0, 30086, 42, 16, 384, 5471, 13734, 20206, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [32]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [33]:
# tokenizeing the entire dataset
def tokenize(batch):
  return tokenizer(
      batch['title'],
      batch["description"],
      padding=True,# this adds a padding to the text to make it of same length
      truncation=True,# Bert has a input size of 512 is the text exceedes that we have to reduce it
      max_length = 512
  )

In [34]:
dataset = data.map(tokenize,batched = True)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [35]:
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'description', 'input_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['label', 'title', 'description', 'input_ids', 'attention_mask'],
        num_rows: 7600
    })
})

In [36]:
print(dataset["train"][0].keys())

dict_keys(['label', 'title', 'description', 'input_ids', 'attention_mask'])


In [37]:
print(len(dataset["train"][0]["input_ids"]))
print(len(dataset["train"][1]["input_ids"]))

318
318


In [38]:
dataset = dataset.remove_columns(
    ["title", "description"]
)

dataset = dataset.rename_column(
    "label",
    "labels"
)

In [39]:
print(dataset["train"][0].keys())

dict_keys(['labels', 'input_ids', 'attention_mask'])


# For now we are directly downloading the model with the classification head in a function

In [40]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    # "bert-base-uncased",
    "roberta-base",
    num_labels=4
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [44]:
print(model.device)
# print(torch.cuda.get_device_name(0))

cpu


In [47]:
import torch
torch.cuda.is_available()
device = torch.device("cuda")

model = model.to(device)

print(model.device)

cuda:0


In [48]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs = 3,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 32,
    learning_rate = 2e-5,
    eval_strategy ="epoch"
)

In [49]:
# trainer api
from transformers import Trainer
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = dataset['train'],
    eval_dataset = dataset['test'],
    data_collator=data_collator
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
